# CourtLens — full local pipeline on Kaggle GPU
Same path as local: YOLO+ByteTrack (stride 5) → kit/jersey identity → possession events → match_stats.
Requires Accelerator = **GPU T4** in notebook settings.

In [ ]:
import glob, subprocess, sys, zipfile, shutil, os
from pathlib import Path

root = Path("/kaggle/input")
print("INPUT TREE:")
for p in sorted(root.rglob("*"))[:80]:
    if p.is_file():
        print(" ", p, p.stat().st_size)

wheels = sorted(glob.glob(str(root / "**/*.whl"), recursive=True))
assert wheels, "No wheels in dataset"
print("wheels", len(wheels))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *wheels])

# CourtLens package: directory zipped by Kaggle as courtlens_offline.zip, or loose .py
pkg_dir = Path("/kaggle/working/courtlens_offline")
if pkg_dir.exists():
    shutil.rmtree(pkg_dir)
pkg_dir.mkdir(parents=True)

zips = sorted(glob.glob(str(root / "**/courtlens_offline.zip"), recursive=True))
zips += sorted(glob.glob(str(root / "**/*courtlens*.zip"), recursive=True))
py_runs = sorted(glob.glob(str(root / "**/run.py"), recursive=True))
print("zips", zips)
print("run.py", py_runs)

if zips:
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(pkg_dir)
elif py_runs:
    src = Path(py_runs[0]).parent
    for f in src.glob("*.py"):
        shutil.copy2(f, pkg_dir / f.name)
else:
    raise SystemExit("courtlens_offline package not found in dataset input")

if not (pkg_dir / "run.py").exists():
    nested = list(pkg_dir.glob("**/run.py"))
    assert nested, f"run.py missing after extract; contents={list(pkg_dir.rglob('*'))}"
    # flatten nested extract
    for f in nested[0].parent.glob("*.py"):
        shutil.copy2(f, pkg_dir / f.name)

print("package files", sorted(p.name for p in pkg_dir.glob("*.py")))
subprocess.call(["bash", "-lc", "tesseract --version | head -1 || echo 'no tesseract binary'"])
print("install done")


In [ ]:
import sys, torch, shutil, subprocess
print("torch", torch.__version__)
cuda = torch.cuda.is_available()
print("cuda available", cuda)
if cuda:
    print("device", torch.cuda.get_device_name(0))
else:
    smi = shutil.which("nvidia-smi")
    if smi:
        subprocess.call([smi])
    else:
        print("nvidia-smi not present (CPU session)")
    print("FATAL: GPU required. In Kaggle UI: Settings → Accelerator → GPU T4, then re-run.")
    sys.exit(1)


In [ ]:
import glob, os, sys
from pathlib import Path

pkg = Path("/kaggle/working/courtlens_offline")
if not (pkg / "run.py").exists():
    pkg = next(Path("/kaggle/working").glob("**/run.py")).parent
sys.path.insert(0, str(pkg))

video = glob.glob("/kaggle/input/**/test_video_2.mp4", recursive=True)[0]
weights = glob.glob("/kaggle/input/**/yolov8n.pt", recursive=True)[0]
os.environ["COURTLENS_VIDEO"] = video
os.environ["COURTLENS_WEIGHTS"] = weights
os.environ["COURTLENS_OUT"] = "/kaggle/working/courtlens_out"
os.environ["VISION_FRAME_STRIDE"] = "5"
# Full video on GPU (run.py sets VISION_DEVICE from torch.cuda)

print("video", video)
print("weights", weights)
import run as courtlens_run
courtlens_run.main()